# Ideogram 4.0 Colab Visual Asset & Typography Generation Pipeline

此 Notebook 專為在 **Google Colab (A100 / L4 / T4)** 上執行 **Ideogram 4.0 (9.3B DiT)** 視覺資產生成而設計。
- **地表最強文字渲染**：專精於精確招牌、標誌、文字排版與電影標題設計，OCR 準確度高達 0.97。
- **NF4 / FP8 量化支援**：NF4 版僅需約 10~12GB VRAM，在 Colab 上運作極為順暢。
- 可作為獨立高品質視覺海報產出，亦可作為 LTX-2.5 影片生成的第一影格（首格圖）視覺資產。

## 1. 檢查 GPU 硬體與資源

In [ ]:
import os, shutil, subprocess, sys, torch
from pathlib import Path

assert torch.cuda.is_available(), "錯誤：未偵測到 GPU，請至「執行階段 > 變更執行階段類型」選擇 GPU。"
gpu_name = torch.cuda.get_device_name(0)
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)
free_disk_gb = round(shutil.disk_usage('/content').free / 2**30, 1)

print(f"[GPU] {gpu_name} (VRAM: {vram_gb} GiB)")
print(f"[CUDA] {torch.version.cuda}")
print(f"[可用磁碟空間] {free_disk_gb} GiB")

## 2. 安裝 Ideogram 4.0 依賴環境

In [ ]:
!pip install -q --upgrade pip
# 安裝最新支援 Ideogram4Pipeline 的 diffusers 與相關套件
!pip install -q "git+https://github.com/huggingface/diffusers.git"
!pip install -q "transformers>=4.48.0" accelerate sentencepiece bitsandbytes torchvision huggingface_hub protobuf
print("Ideogram 4.0 依賴安裝完成。")

## 3. Hugging Face 認證與快取配置

In [ ]:
hf_token = os.environ.get("HF_TOKEN", "").strip()
if hf_token:
    from huggingface_hub import login
    login(token=hf_token, add_to_git_credential=False)
    print("Hugging Face 授權完成。")

cache_dir = "/content/cache_ideogram"
use_drive_cache = os.environ.get("USE_DRIVE_CACHE", "1") == "1"
if use_drive_cache:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        drive_path = Path('/content/drive/MyDrive/models/Ideogram-4')
        drive_path.mkdir(parents=True, exist_ok=True)
        cache_dir = str(drive_path)
        print(f"啟用 Google Drive 快取: {cache_dir}")
    except Exception as e:
        print(f"略過 Drive 掛載: {e}")
        os.makedirs(cache_dir, exist_ok=True)
else:
    os.makedirs(cache_dir, exist_ok=True)

## 4. 讀取任務參數與提示詞

In [ ]:
prompt_file = os.environ.get("IDEOGRAM_PROMPT_FILE", "/content/prompt.txt")
output_path = os.environ.get("IDEOGRAM_OUTPUT_PATH", "/content/output.png")
width = int(os.environ.get("IDEOGRAM_WIDTH", "1280"))
height = int(os.environ.get("IDEOGRAM_HEIGHT", "720"))
steps = int(os.environ.get("IDEOGRAM_STEPS", "30")) # 預設 20~30 步兼顧品質與速度
seed = int(os.environ.get("IDEOGRAM_SEED", "42"))
model_name = os.environ.get("IDEOGRAM_MODEL", "ideogram-ai/ideogram-4-nf4")

if os.path.exists(prompt_file):
    with open(prompt_file, "r", encoding="utf-8") as f:
        prompt = f.read().strip()
else:
    prompt = os.environ.get("IDEOGRAM_PROMPT", "A cinematic vintage neon sign glowing in a dark cyberpunk alley that reads 'NEON HORIZON' with vivid electric blue and pink reflections, 8k")

print("=== Ideogram 4.0 任務配置 ===")
print(f"Prompt: {prompt}")
print(f"解析度: {width}x{height}")
print(f"推論步數: {steps}")
print(f"隨機種子: {seed}")
print(f"模型: {model_name}")
print(f"輸出路徑: {output_path}")

## 5. 載入 Ideogram 4.0 模型並執行推論

In [ ]:
import torch
from diffusers import Ideogram4Pipeline

print(f"正在載入 {model_name} (請稍候)...\n")
pipe = Ideogram4Pipeline.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    cache_dir=cache_dir
)
pipe.to("cuda")

generator = torch.Generator(device="cuda").manual_seed(seed)

print("開始 Ideogram 4.0 推論 (文字排版精準渲染中)...")
image = pipe(
    prompt=prompt,
    width=width,
    height=height,
    num_inference_steps=steps,
    generator=generator
).images[0]

Path(output_path).parent.mkdir(parents=True, exist_ok=True)
image.save(output_path)
print(f"\n視覺資產生成完畢！已儲存至: {output_path}")

## 6. 驗證產出成品

In [ ]:
assert os.path.exists(output_path), f"生成失敗：未找到輸出檔案 {output_path}"
size_kb = round(os.path.getsize(output_path) / 1024, 1)
print(f"[檔案大小] {size_kb} KB")
print("狀態：Ideogram 4.0 視覺資產生成與驗證成功！")